In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')

import os
import time
import numpy as np
from itertools import chain
import pickle
from glob import glob
from pathlib import Path
from copy import deepcopy

from multiprocessing import Pool

from cylinder_extraction.station import Station
from cylinder_alignment.station_alignment import StationAlignment
from utils.write_output import write_array2txt
from utils.math_methods import project_point_onto_plane

from cylinder_extraction.utils import detect_objects

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## For multiple directories

In [3]:
scan_field_names = [
    ('JB-448 26ft EXT Faro',8),
    ('JB-448 26ft EXT Faro/tilted stations', 8),
    ('JB-567 27ft EXT Leica/tilted stations',8),
    ('JB-531 40ft EXT',8),
    ('JB-453 60ft EXT Faro',8),
    ('JB-595 117ft EXT - FARO',8),
    ('JB-550 90ft EXT Trimble',2),
    ('JB-505 100ft EXT Trimble',2),
    ('JB-566 120ft EXT',8)
]
execuated_time = {}
for field, pack_num in scan_field_names[8:9]:
    time_begin = time.time()
    try:
        print('===========================')
        print(field)
        data_dir = Path(f'D:/REGISTRATION DATA/External/{field}')
        filename_template = '{}.e57'
        ptc_col_names =  ["x","y","z"]

        source_files = glob(str(data_dir / '*.e57')) + glob(str(data_dir / '*.pcd'))
        station_names = []
        stations = []
        for ind, fp in enumerate(source_files):
            if 'roof' in fp.lower():
                continue
            # station name
            station_name = Path(fp).stem
            # station id
            station_id = ind

            print(f'STATION {station_name}')
            station = Station(
                name=station_name, file_path=fp, read_points=False, station_id=station_id
            )
            stations.append(station)

        # pack_num = 8
        num_stations = len(stations)
        for i in range(0, num_stations, pack_num):
            j = min(i+pack_num, num_stations)
            pack = stations[i:j]
            with Pool(pack_num) as p:
                pack = p.map(
                    detect_objects,
                    pack
                )
            for station in pack:
                station.clusters = None
                # write station metadata
                out_dir = os.path.join(data_dir,station.name)
                if not os.path.exists(out_dir):
                    os.mkdir(out_dir)
                # save station without points
                station.ptc = None
                out_fp = os.path.join(out_dir, f'{station.name}.pkl')
                with open(out_fp, 'wb') as f:
                    pickle.dump(station, f)
    except Exception as e:
        print(f'Failed handling scan {field}')
        print(e)
    time_end = time.time()
    lasted = time_end - time_begin
    print(f'processing took {lasted} seconds')
    execuated_time[field] = lasted
        

JB-566 120ft EXT
STATION Plains Tank 401- Setup 014
STATION Plains Tank 401- Setup 015
STATION Plains Tank 401- Setup 016
STATION Plains Tank 401- Setup 017
STATION Plains Tank 401- Setup 018
STATION Plains Tank 401- Setup 019
STATION Plains Tank 401- Setup 020
STATION Plains Tank 401- Setup 021
STATION Plains Tank 401- Setup 022
STATION Plains Tank 401- Setup 023
STATION Plains Tank 401- Setup 024
STATION Plains Tank 401- Setup 025
STATION Plains Tank 401- Setup 026
STATION Plains Tank 401- Setup 027
STATION Plains Tank 401- Setup 028
STATION Plains Tank 401- Setup 029
STATION Plains Tank 401- Setup 030
processing took 893.779714345932 seconds


In [4]:
execuated_time

{'JB-566 120ft EXT': 893.779714345932}

In [9]:
data_dir

WindowsPath('D:/REGISTRATION DATA/External/JB-448 26ft EXT Faro/tilted stations')

In [10]:
station = Station(
    name='89', 
    file_path='D:\REGISTRATION DATA\External\JB-448 26ft EXT Faro\\tilted stations\Jb-448 Scan_FARO_LS03550_089.e57',
    read_points=True, station_id=89
)

D:\REGISTRATION DATA\External\JB-448 26ft EXT Faro\tilted stations Jb-448 Scan_FARO_LS03550_089.e57
[ 5.23032625e-01 -5.81092281e-04  4.88920343e-02 -8.50908987e-01]
[ -1.05777655   1.05632521 -20.15590082]
fields
['cartesianX', 'cartesianY', 'cartesianZ', 'intensity', 'colorRed', 'colorGreen', 'colorBlue']


In [ ]:
station.extract_cylinder()